In [1]:
import numpy as np

In [2]:
n = 100_000
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=n)

wt = 1  # Can modify this to take an exponentially weighted moving average
wtSum = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = np.sqrt(100_000_000) * np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    wtSum = wt * wtSum + 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / wtSum
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M = wt * M + delta @ new_delta
    Sigma_hat = M / wtSum

    # Estimation of the inverse of Sigma
    Mnum = 1.0 / np.power(wt, 2) * Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + 1.0 / wt * new_delta @ Minv @ delta
    Minv = Minv / wt - Mnum / Mden.item()

    Chol = Chol 

print("Error between estimates (should be very small): ")
print(np.linalg.pinv(Sigma_hat) - Minv * wtSum)

print("Error between inverse of real and estimated covariance matrix:")
print(np.linalg.pinv(Sigma) - Minv * wtSum)

Error between estimates (should be very small): 
[[ 7.16982029e-13 -1.06115117e-12  8.38745740e-13]
 [ 1.27098332e-12 -1.06858966e-12  5.70127279e-13]
 [-7.64749375e-13  1.02587383e-12 -4.65960603e-13]]
Error between inverse of real and estimated covariance matrix:
[[0.0025169  0.00425884 0.00152781]
 [0.00425884 0.00275907 0.00158127]
 [0.00152781 0.00158127 0.00022925]]


That's great, but we don't just want to know the inverse covariance. When we use the inverse covariance, we use the _square root_ of it in normalization.

So what we really want is to know $M$, where $MM^\top = \Sigma^{-1}$.

Let's begin by writing out an update to an estimate of $\Sigma$, which we will call $S$.

$S + uv^\top$

This update is the online formula for covariance estimation, where $u = x_t - \hat{\mu}_{t-1}$ and $v = x_t - \hat{\mu}_t$. [See Wikipedia for details.](https://en.wikipedia.org/wiki/Algorithms_for_calculating_variance#Online), but it was the formulation I used above.

We want to know the inverse of this (and eventually the square root of this inverse):

$(S + uv^\top)^{-1} = S^{-1} - \frac{S^{-1} uv^\top S^{-1}}{1 + v^\top S^{-1} u}$

By the Sherman-Morrison formula. Suppose, however, that $S = LL^\top$, because it is positive definite, as a covariance matrix. Then we can write this as:

$(LL^\top)^{-1} - \frac{(LL^\top)^{-1} uv^\top (LL^\top)^{-1}}{1 + v^\top (LL^\top)^{-1} u}$

If we do a bit of algebra, we can pull out a term on the left and a corresponding one on the right:

$(L^{-1})^\top \left[ I - \frac{L^{-1} uv^\top (L^{-1})^\top}{1 + v^\top (L^{-1})^\top L^{-1} v}\right] L^{-1}$

Now call $x = L^{-1} u$ and $y = L^{-1} v$, just a rotated version of the original vectors, $u$ and $v$.

$(L^{-1})^\top \left[ I - \frac{xy^\top}{1 + y^\top x}\right] L^{-1}$.

Critically, the inner quantity, $I - \frac{xy^\top}{1 + y^\top x}$ admits a simple rank-one update to the cholesky of $I$.

So to update the Cholesky of the inverse ($L^{-1}$), we simply take this (easy to compute) Cholesky of the inner quantity. Apparently this is $O(d)$ time complexity using an algorithm of [Krause and Igel (2015)](https://dl.acm.org/doi/10.1145/2725494.2725496) (and [exposed in Tensorflow](https://www.tensorflow.org/probability/api_docs/python/tfp/math/cholesky_update), at least), which is cool, albeit kind of wasted, because we're still going to have to multiply $L^{-1}$ and the cholesky of the inner bit.

In [ ]:
n = 100_000
d = 3

mu = np.array([1, 10, 20]).reshape(-1, 1)
Sigma = np.array([1, -1, 0, -1, 3, -1, 0, -1, 3]).reshape(3, 3)

X = np.random.multivariate_normal(mean=mu.flatten(), cov=Sigma, size=n)

n = 0.0
muhat = np.zeros((3, 1))

# Starting estimates of covariance / inverse covariance.
M = 0 * np.eye(d)
Minv = 100_000_000 * np.eye(d)
Chol = np.sqrt(100_000_000) * np.eye(d)

for x in X:
    x = x.reshape(-1, 1)
    n += 1  # Divisor For Covariance Matrix
    delta = x - muhat
    muhat += delta / n
    new_delta = (x - muhat).reshape(1, -1)

    # Estimation of Sigma
    M = M + delta @ new_delta
    Sigma_hat = M / n

    # Estimation of the inverse of Sigma
    Mnum = Minv @ delta @ new_delta @ Minv
    Mden = 1.0 + new_delta @ Minv @ delta
    Minv = Minv - Mnum / Mden.item()

    Chol = Chol 

print("Error between estimates (should be very small): ")
print(np.linalg.pinv(Sigma_hat) - Minv * wtSum)

print("Error between inverse of real and estimated covariance matrix:")
print(np.linalg.pinv(Sigma) - Minv * wtSum)